# 🧠 GlioRad: Lower-Grade Glioma Brain MRI Segmentation — Starter Guide

Welcome to the **GlioRad Benchmark Challenge**!
This challenge tasks participants with performing accurate semantic segmentation of FLAIR abnormality lesions in brain MRI scans from the TCGA-LGG cohort.

### Official Dataset Source:
- **Dataset:** [Brain MRI Segmentation (`mateuszbuda/lgg-mri-segmentation`) on Kaggle](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation)
- **Original Source:** The Cancer Genome Atlas (TCGA) & The Cancer Imaging Archive (TCIA)
- **Citation:** Buda et al., *Computers in Biology and Medicine*, 2019
- **Evaluation Metric:** Mean Slice Dice Similarity Coefficient ($\text{DSC}$)

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Setup project paths
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
public_data_dir = project_root / 'data' / 'public'
starter_dir = project_root / 'starter_code'
competition_dir = project_root / 'competition'

print(f'Project root: {project_root}')
print(f'Public data directory: {public_data_dir}')

## 1. Inspect Public Benchmark Manifests
The repository distributes public manifests with strict zero-leakage patient-level partitioning:
- `train_manifest.csv`: Slices with patient IDs, tumor status, and genomic markers (IDH mutation, 1p/19q codeletion).
- `test_manifest.csv`: Completely held-out test patient slices without ground-truth masks.
- `sample_submission.csv`: Schema-compliant Run-Length Encoded (RLE) candidate prediction format.

In [ ]:
train_manifest = pd.read_csv(public_data_dir / 'train_manifest.csv')
test_manifest = pd.read_csv(public_data_dir / 'test_manifest.csv')
sample_sub = pd.read_csv(public_data_dir / 'sample_submission.csv')

print(f'Training Slices: {len(train_manifest):,} across {train_manifest["patient_id"].nunique()} patients')
print(f'Test Slices: {len(test_manifest):,} held-out slices to evaluate')
print(f'Sample Submission rows: {len(sample_sub):,}')
train_manifest.head()

## 2. Exploratory Data Analysis & Radiogenomics Visualizer
Run the official visualization dashboard from `starter_code/visualize_data.py` to examine patient tumor burdens, IDH mutation distributions, and slice depth frequency.

In [ ]:
sys.path.append(str(starter_dir))
from visualize_data import create_tcga_lgg_eda_dashboard

search_dirs = [project_root / 'data' / 'kaggle_lgg', project_root / 'data' / 'processed_lgg']
eda_output = starter_dir / 'eda_tcga_lgg_overview.png'

create_tcga_lgg_eda_dashboard(
    manifest_csv=public_data_dir / 'train_manifest.csv',
    search_dirs=search_dirs,
    output_path=eda_output,
    show_plot=True
)

## 3. Run the Starter Baseline Segmentation
We provide a starter baseline in `starter_code/baseline.py`. It reads `test_manifest.csv` and outputs candidate RLE abnormality masks.

In [ ]:
import subprocess

baseline_script = starter_dir / 'baseline.py'
result = subprocess.run([sys.executable, str(baseline_script)], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

## 4. Validate Submission Format & RLE Encoding
Ensure your submission passes all verification checks (patient ID matches, slice index completeness, valid Run-Length Encoded string format).

In [ ]:
validate_script = competition_dir / 'validate_submission.py'
sub_file = project_root / 'submissions' / 'baseline_submission.csv'
test_file = public_data_dir / 'test_manifest.csv'
lb_file = project_root / 'leaderboard' / 'leaderboard.csv'

if sub_file.exists():
    res = subprocess.run([sys.executable, str(validate_script), str(sub_file), str(test_file), str(lb_file)], capture_output=True, text=True)
    print(res.stdout)
else:
    print(f'{sub_file} not found. Run baseline.py first.')

## 5. Encrypt Submission for Pull Request
Encrypt your predictions with the official challenge 2048-bit RSA public key (`starter_code/public_key.pem`).

In [ ]:
encrypt_script = starter_dir / 'encrypt.py'
pub_key = starter_dir / 'public_key.pem'
enc_output = project_root / 'submissions' / 'baseline_submission.csv.enc'

if sub_file.exists():
    res = subprocess.run([sys.executable, str(encrypt_script), str(sub_file), str(pub_key), str(enc_output)], capture_output=True, text=True)
    print(res.stdout)
    print(f'✅ Ready to submit! Commit and push {enc_output} in a Pull Request.')
else:
    print('Generate a submission CSV before encrypting.')

## 🚀 Training on Kaggle with GPU
For full deep learning model training (U-Net, Attention U-Net, Swin-UNETR) on the complete 110-patient dataset:
1. Open `notebooks/kaggle_tcga_lgg_training.ipynb`.
2. Upload it to [Kaggle](https://www.kaggle.com/code).
3. In the right-hand sidebar under **Input**, click **`+ Add Input`** and search for:
   👉 `mateuszbuda/lgg-mri-segmentation`
4. Enable GPU accelerator (GPU T4 x2) and train your segmentation model!